# Module 5: RAG Pipeline with Azure DocumentDB

**Time**: ~60 min  
**Environment**: Jupyter notebook in VS Code

This notebook is fully runnable. Enter your Azure DocumentDB connection string and OpenAI API key in Step 0, then run each cell in order. The notebook creates embeddings for RAG chunks, stores them in Azure DocumentDB, retrieves context with vector and hybrid search, and builds a grounded prompt for a chat model.

The final step prints the prompt your application would send to a chat model.


## Step 0: Connect and configure embeddings

This cell loads the MongoDB driver, accepts the DocumentDB connection string and OpenAI API key, and opens the RAG chunks collection.

In [ ]:
let mongodb;
try { mongodb = require("mongodb"); } catch { require("child_process").execSync("npm install mongodb", { stdio: "inherit" }); mongodb = require("mongodb"); }
const { MongoClient } = mongodb;
const connectionString = process.env.DOCUMENTDB_CONNECTION_STRING || "<paste-your-azure-documentdb-connection-string-here>";
const openAiApiKey = process.env.OPENAI_API_KEY || "<paste-your-openai-api-key-here>";
const embeddingModel = process.env.OPENAI_EMBEDDING_MODEL || "text-embedding-3-small";
if (connectionString.includes("<paste")) throw new Error("Paste your Azure DocumentDB connection string in this cell or set DOCUMENTDB_CONNECTION_STRING.");
if (openAiApiKey.includes("<paste")) throw new Error("Paste your OpenAI API key in this cell or set OPENAI_API_KEY.");
const client = new MongoClient(connectionString);
await client.connect();
const db = client.db("docdbworkshop");
const chunks = db.collection("rag_chunks");
await db.command({ ping: 1 });

## Step 1: Generate embeddings for chunks

Each chunk is embedded with OpenAI and inserted into Azure DocumentDB.

In [ ]:
async function createEmbedding(text) {
  const response = await fetch("https://api.openai.com/v1/embeddings", { method: "POST", headers: { "Authorization": `Bearer ${openAiApiKey}`, "Content-Type": "application/json" }, body: JSON.stringify({ model: embeddingModel, input: text }) });
  if (!response.ok) throw new Error(await response.text());
  return (await response.json()).data[0].embedding;
}
const ragDocs = [
  { _id: "rag-001", sourceId: "search-module", title: "Vector search", chunk: "Azure DocumentDB vector search uses the $search stage with the cosmosSearch operator to retrieve documents by embedding similarity.", url: "module-4-search", tags: ["vector", "search"] },
  { _id: "rag-002", sourceId: "search-module", title: "Full-text search", chunk: "Azure DocumentDB full-text search uses createSearchIndexes and the $search text operator to return BM25-ranked keyword matches.", url: "module-4-search", tags: ["full-text", "bm25"] },
  { _id: "rag-003", sourceId: "search-module", title: "Hybrid search", chunk: "Hybrid search runs BM25 keyword retrieval and vector retrieval, then combines ranked lists with Reciprocal Rank Fusion.", url: "module-4-search", tags: ["hybrid", "rrf"] },
  { _id: "rag-004", sourceId: "rag-module", title: "Grounded generation", chunk: "A RAG pipeline retrieves relevant chunks from Azure DocumentDB and includes them in the model prompt so the answer is grounded in current application data.", url: "module-5-rag", tags: ["rag", "generation"] }
];
await chunks.drop().catch(() => {});
for (const doc of ragDocs) doc.embedding = await createEmbedding(doc.chunk);
await chunks.insertMany(ragDocs);
const embeddingDimensions = ragDocs[0].embedding.length;
({ loaded: await chunks.countDocuments({}), embeddingDimensions });

## Step 2: Create retrieval indexes

Create a DiskANN vector index and a BM25 search index on the same chunk collection.

**STUDENT EXERCISE:** you will complete the index command in the next cell. Compare with the matching `after` notebook if you get stuck.


In [ ]:
// STUDENT EXERCISE: create idx_chunk_embedding_diskann and idx_chunk_fts on rag_chunks.
const vectorIndexResult = { ok: 0, message: "TODO: create idx_chunk_embedding_diskann" };
const fullTextIndexResult = { ok: 0, message: "TODO: create idx_chunk_fts" };
({ vectorIndexResult, fullTextIndexResult });


## Step 3: Generate a question embedding and retrieve context

The question is embedded at runtime and used by `cosmosSearch` to retrieve nearest chunks.

**STUDENT EXERCISE:** complete the vector retrieval query. Expected result: top chunks/documents should relate to RAG, vector, or hybrid search.


In [ ]:
const question = "How does DocumentDB retrieve context for RAG?";
const questionVector = await createEmbedding(question);
// STUDENT EXERCISE: retrieve the top 3 chunks with $search.cosmosSearch.
const vectorContext = [];
vectorContext;


## Step 4: Hybrid retrieval with RRF

Hybrid retrieval combines BM25 and vector candidates for better grounding coverage.

**STUDENT EXERCISE:** complete the search or hybrid retrieval snippet in the next cell. Notice where `$limit` belongs and how `searchScore` is projected.


In [ ]:
function rrf(lists, k = 60, topN = 3) { const scores = new Map(), docsById = new Map(); for (const list of lists) list.forEach((doc, rank) => { const id = doc._id.toString(); docsById.set(id, doc); scores.set(id, (scores.get(id) ?? 0) + 1 / (k + rank + 1)); }); return [...scores.entries()].sort((a,b)=>b[1]-a[1]).slice(0,topN).map(([id,score])=>({ ...docsById.get(id), rrfScore: score })); }
// STUDENT EXERCISE: fill keywordContext with $search.text, then fuse with vectorContext.
const keywordContext = [];
const hybridContext = rrf([keywordContext, vectorContext]);
hybridContext;


## Step 5: Build the grounded prompt

The final prompt includes retrieved chunks and instructs the model not to answer outside the provided context.

In [ ]:
// STUDENT EXERCISE: build a grounded prompt from hybridContext using <context> tags.
const groundedPrompt = "TODO: build grounded prompt";
groundedPrompt;
